In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("../data/dormmatch_cleaned.csv")
print("Original dataset:", df.shape)
df.head()

Original dataset: (2000, 24)


,student_id,age,gender,year_of_study,branch,cgpa,sleep_time,wake_time,sleep_duration,cleanliness_level,...,sharing_preference,weekend_activity,food_habit,smoking_preference,ac_preference,preferred_room_type,budget,preferred_environment,roommate_interaction,calculated_sleep_duration
0,STU0760,19,Female,3,Biotechnology,9.03,2.35,6.71,4.07,3.0,...,3,Outings,Vegetarian,No smoking,Yes,Double,6500.0,Balanced,2,4.36
1,STU1501,21,Male,1,Electrical Engineering,7.92,23.50,6.98,7.31,4.0,...,5,Gaming,Vegetarian,No smoking,Yes,Triple,6500.0,Balanced,4,7.48
2,STU1385,20,Female,4,Computer Science,8.93,22.99,7.18,8.47,5.0,...,3,Movies,non-vegetarian,No smoking,No,Double,6500.0,Social,2,8.19
3,STU0052,20,Male,2,Computer Science,7.45,22.25,5.81,7.90,4.0,...,3,Movies,Eggetarian,No smoking,Yes,Single,5500.0,quiet,4,7.56
4,STU0660,20,Female,2,Electrical Engineering,7.02,0.77,10.68,10.38,2.0,...,5,Relaxing,Non-vegetarian,No preference,Yes,Double,6000.0,Balanced,3,9.91


In [2]:
# Encode sleep and wake times on the circular 24-hour clock.
df["sleep_time"] = df["sleep_time"] % 24
df["wake_time"] = df["wake_time"] % 24

df["sleep_time_sin"] = np.sin(2 * np.pi * df["sleep_time"] / 24)
df["sleep_time_cos"] = np.cos(2 * np.pi * df["sleep_time"] / 24)
df["wake_time_sin"] = np.sin(2 * np.pi * df["wake_time"] / 24)
df["wake_time_cos"] = np.cos(2 * np.pi * df["wake_time"] / 24)

print("Circular time features added")

Circular time features added


In [3]:
# Exploratory categories retain meaningful labels but are not added automatically to clustering.
df["study_intensity"] = pd.cut(
    df["study_hours"],
    bins=[-np.inf, 2, 5, 8, np.inf],
    labels=["Low", "Moderate", "High", "Very High"],
)

df["sleep_category"] = pd.cut(
    df["sleep_duration"],
    bins=[0, 5, 7, 9, np.inf],
    labels=["Short", "Moderate", "Ideal", "Long"],
    include_lowest=True,
)

df["social_category"] = pd.cut(
    df["social_activity"],
    bins=[0, 2, 3, 5],
    labels=["Low", "Medium", "High"],
    include_lowest=True,
)

df[["study_hours", "study_intensity", "sleep_duration", "sleep_category", "social_category"]].head()

,study_hours,study_intensity,sleep_duration,sleep_category,social_category
0,2.67,Moderate,4.07,Short,Medium
1,2.83,Moderate,7.31,Ideal,High
2,5.41,High,8.47,Ideal,Low
3,3.63,Moderate,7.90,Ideal,Low
4,4.54,Moderate,10.38,Long,High


In [4]:
lifestyle_features = [
    "sleep_time_sin", "sleep_time_cos",
    "wake_time_sin", "wake_time_cos",
    "sleep_duration",
    "cleanliness_level", "noise_tolerance",
    "study_hours", "social_activity", "introvert_extrovert",
    "sharing_preference", "weekend_activity", "food_habit",
    "smoking_preference", "ac_preference", "preferred_room_type",
    "preferred_environment", "roommate_interaction",
]

# Ordinal 1-5 fields remain numeric rather than becoming unrelated dummy categories.
ordinal_columns = ["sharing_preference", "roommate_interaction"]
for column in ordinal_columns:
    df[column] = pd.to_numeric(df[column], errors="coerce")

print("Lifestyle features:", len(lifestyle_features))

Lifestyle features: 18


In [5]:
df_encoded = pd.get_dummies(
    df[lifestyle_features],
    drop_first=False,
    dtype=int,
)

print("Encoded dataset:", df_encoded.shape)
df_encoded.head()

Encoded dataset: (2000, 47)


,sleep_time_sin,sleep_time_cos,wake_time_sin,wake_time_cos,sleep_duration,cleanliness_level,noise_tolerance,study_hours,social_activity,introvert_extrovert,...,ac_preference_Yes,preferred_room_type_Double,preferred_room_type_Single,preferred_room_type_Triple,preferred_environment_Balanced,preferred_environment_Quiet,preferred_environment_Social,preferred_environment_balanced,preferred_environment_quiet,preferred_environment_social
0,0.577145,0.816642,0.982774,-0.184809,4.07,3.0,4.0,2.67,3,3.0,...,1,1,0,0,1,0,0,0,0,0
1,-0.130526,0.991445,0.967268,-0.253758,7.31,4.0,3.0,2.83,4,4.0,...,1,0,0,1,1,0,0,0,0,0
2,-0.261347,0.965245,0.952661,-0.304033,8.47,5.0,1.0,5.41,2,1.0,...,0,1,0,0,0,0,1,0,0,0
3,-0.442289,0.896873,0.998763,0.049721,7.90,4.0,2.0,3.63,1,1.0,...,1,0,1,0,0,0,0,0,1,0
4,0.200223,0.979750,0.338738,-0.940881,10.38,2.0,5.0,4.54,4,5.0,...,1,1,0,0,1,0,0,0,0,0


In [6]:
# Compare two valid scaling strategies for K-Means.
scaler_all = StandardScaler()
X_scaled_all = pd.DataFrame(
    scaler_all.fit_transform(df_encoded),
    columns=df_encoded.columns,
    index=df_encoded.index,
)

# Keep binary one-hot columns unscaled in the mixed approach.
continuous_columns = [
    "sleep_time_sin", "sleep_time_cos",
    "wake_time_sin", "wake_time_cos",
    "sleep_duration", "cleanliness_level", "noise_tolerance",
    "study_hours", "social_activity", "introvert_extrovert",
    "sharing_preference", "roommate_interaction",
]
continuous_columns = [column for column in continuous_columns if column in df_encoded.columns]
scaler_mixed = StandardScaler()
X_scaled_mixed = df_encoded.copy()
X_scaled_mixed[continuous_columns] = scaler_mixed.fit_transform(
    X_scaled_mixed[continuous_columns]
)

print("All-features scaled:", X_scaled_all.shape)
print("Numerical-scaled mixed matrix:", X_scaled_mixed.shape)

X_scaled_df = X_scaled_mixed

All-features scaled: (2000, 47)
Numerical-scaled mixed matrix: (2000, 47)


In [7]:
df_features = df.copy()

df_features.to_csv("../data/dormmatch_feature_engineered.csv", index=False)
df_encoded.to_csv("../data/dormmatch_encoded_features.csv", index=False)
X_scaled_df.to_csv("../data/dormmatch_scaled_features.csv", index=False)

print("Original dataset:", df.shape)
print("Encoded dataset:", df_encoded.shape)
print("Scaled dataset:", X_scaled_df.shape)
print("Missing values:", df_encoded.isna().sum().sum())
print("Infinite values:", np.isinf(df_encoded.select_dtypes(include=np.number)).sum().sum())
print("Duplicate columns:", df_encoded.columns.duplicated().sum())

Original dataset: (2000, 31)
Encoded dataset: (2000, 47)
Scaled dataset: (2000, 47)
Missing values: 0
Infinite values: 0
Duplicate columns: 0
